## SQL for Data People

## Part A — 20 SQL queries (the deliverable)

## TASK 01 Twenty queries of increasing difficulty

In [1]:
import pandas as pd
import sqlite3

#Load your CSV files
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

#Create the SQLite database connection
conn = sqlite3.connect("sales.db")


#Load the DataFrames into SQLite
sales.to_sql("sales", conn, if_exists="replace", index=False)


#Run SQL query
sql = """
SELECT *
FROM sales;

"""

result = pd.read_sql_query(sql, conn)
print(result.head())


#-----------------------------------------------------------------------

#Q1: Show the first 10 rows of the sales table.

sql = """
SELECT *
FROM sales
LIMIT 10;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q2: Count the total number of sales records (orders)
sql = """
SELECT COUNT(*) AS total_orders
FROM sales;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q3: Show all sales records with a quantity of 5
sql ="""SELECT *
FROM sales
WHERE quantity = 5;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q4: List all products in the Electronics category
sql = """
SELECT *
FROM PRODUCTS
WHERE category = 'Electronics' ;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q5: Show the 5 most expensive products
sql="""
SELECT *
FROM products
ORDER BY unit_price DESC
LIMIT 5;
"""
result = pd.read_sql_query(sql, conn)
display(result)


#// MEDIUM: GROUP BY / AGGREGATES / HAVING

#Q6: Count how many customers are in each region
sql = """
SELECT region, COUNT(*) AS customer_count
FROM customers
Group BY region;
"""

#Q7: Find the total quantity sold for each product
sql = """
SELECT
    p.product_id,
    p.product_name,
    SUM(s.quantity) AS total_quantity_sold
FROM products AS p
LEFT JOIN sales AS s ON p.product_id = s.product_id
GROUP BY p.product_id, p.product_name;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q8: Find the average unit price per category
sql = """
SELECT category, AVG(unit_price) AS average_unit_price
FROM products
GROUP BY category;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q9: Count sales records per product, showing only products with more than 50.
sql = """
SELECT
    p.product_id,
    p.product_name,
    COUNT(s.order_id) AS order_count
FROM products AS p
JOIN sales AS s ON p.product_id = s.product_id
GROUP BY p.product_id, p.product_name
HAVING COUNT(s.order_id) > 50;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q10: Find the highest and lowest unit prices
sql = """
SELECT
    MAX(unit_price) AS highest_price,
    MIN(unit_price) AS lowest_price
FROM products;
"""
result = pd.read_sql_query(sql, conn)
display(result)


#// HARDER: JOINS

#Q11: Show each sale's product name and unit price.
sql ="""
SELECT
    s.order_id,
    p.product_name,
    p.unit_price
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id;
"""
result = pd.read_sql_query(sql, conn)
display(result)


# Q12: Calculate total revenue across all sales
sql = """
SELECT
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q13: Calculate total revenue per customer region.
sql ="""
SELECT
    c.region,
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN customers AS c ON s.customer_id = c.customer_id
JOIN products AS p ON s.product_id = p.product_id
GROUP BY c.region;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q14: Calculate total revenue per product category.
sql = """
SELECT
    p.category,
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN products AS p ON s.product_id = p.product_id
GROUP BY p.category;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#Q15: Count sales referencing customers missing from the customers table.
sql = """
SELECT COUNT(*) AS sales_with_missing_customers
FROM sales AS s
LEFT JOIN customers AS c ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
"""
result = pd.read_sql_query(sql, conn)
display(result)

#// ADVANCED: WINDOWS / CTEs / SUBQUERIES

#Q16: Rank products by total revenue, highest revenue first.
sql = """
WITH product_revenue AS (
    SELECT
        p.product_id,
        p.product_name,
        COALESCE(SUM(s.quantity * p.unit_price), 0) AS total_revenue
    FROM products AS p
    LEFT JOIN sales AS s ON p.product_id = s.product_id
    GROUP BY p.product_id, p.product_name
)
SELECT
    product_id,
    product_name,
    total_revenue,
    RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
FROM product_revenue;
"""
result = pd.read_sql_query(sql, conn)
display(result)


#Q17: Calculate a running total of revenue by month.
sql = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p ON s.product_id = p.product_id
    GROUP BY strftime('%Y-%m', s.date)
)
SELECT
    month,
    revenue,
    SUM(revenue) OVER (ORDER BY month) AS running_total
FROM monthly_revenue
ORDER BY month;
"""
result = pd.read_sql_query(sql, conn)
display(result)


#Q18: Find the top-selling product within each category.
sql = """
WITH product_sales AS (
    SELECT
        p.category,
        p.product_id,
        p.product_name,
        COALESCE(SUM(s.quantity), 0) AS total_quantity_sold
    FROM products AS p
    LEFT JOIN sales AS s ON p.product_id = s.product_id
    GROUP BY p.category, p.product_id, p.product_name
),
ranked_products AS (
    SELECT
        *,
        RANK() OVER (
            PARTITION BY category
            ORDER BY total_quantity_sold DESC
        ) AS category_rank
    FROM product_sales
)
SELECT
    category,
    product_name,
    total_quantity_sold
FROM ranked_products
WHERE category_rank = 1;
"""
result = pd.read_sql_query(sql, conn)
display(result)



#Q19: Find products with revenue above the average product revenue.
sql = """
WITH product_revenue AS (
    SELECT
        p.product_id,
        p.product_name,
        COALESCE(SUM(s.quantity * p.unit_price), 0) AS total_revenue
    FROM products AS p
    LEFT JOIN sales AS s ON p.product_id = s.product_id
    GROUP BY p.product_id, p.product_name
)
SELECT *
FROM product_revenue
WHERE total_revenue > (
    SELECT AVG(total_revenue)
    FROM product_revenue
);
"""
result = pd.read_sql_query(sql, conn)
display(result)


#Q20: Calculate each region's percentage of total revenue.
sql = """
WITH region_revenue AS (
    SELECT
        c.region,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN customers AS c ON s.customer_id = c.customer_id
    JOIN products AS p ON s.product_id = p.product_id
    GROUP BY c.region
)
SELECT
    region,
    revenue,
    ROUND(
        100.0 * revenue / SUM(revenue) OVER (),
        2
    ) AS revenue_percentage
FROM region_revenue
ORDER BY revenue DESC;
"""
result = pd.read_sql_query(sql, conn)
display(result)



  order_id        date customer_id product_id  quantity
0   O00001  31-10-2024        C999       P019         2
1   O00002  14-03-2024        C999       P014         1
2   O00003  21-10-2024        C999       P004         1
3   O00004  03-01-2024        C999       P003         3
4   O00005  18-10-2024        C999       P018         4


,order_id,date,customer_id,product_id,quantity
0,O00001,31-10-2024,C999,P019,2
1,O00002,14-03-2024,C999,P014,1
2,O00003,21-10-2024,C999,P004,1
3,O00004,03-01-2024,C999,P003,3
4,O00005,18-10-2024,C999,P018,4
5,O00006,15-10-2024,C010,P006,2
6,O00007,12-10-2024,C013,P016,5
7,O00008,31-08-2024,C046,P003,2
8,O00009,21-06-2024,C048,P006,5
9,O00010,15-09-2024,C003,P006,2


,total_orders
0,1000


,order_id,date,customer_id,product_id,quantity
0,O00007,12-10-2024,C013,P016,5
1,O00009,21-06-2024,C048,P006,5
2,O00014,16-06-2024,C015,P019,5
3,O00020,11-02-2024,C025,P015,5
4,O00022,01-09-2024,C005,P020,5
...,...,...,...,...,...
186,O00981,02-12-2024,C042,P009,5
187,O00988,28-12-2024,C034,P004,5
188,O00989,26-08-2024,C016,P005,5
189,O00990,21-02-2024,C001,P010,5


,product_id,product_name,category,unit_price
0,P001,Laptop,Electronics,800
1,P002,Mouse,Electronics,25
2,P003,Keyboard,Electronics,45
3,P004,Monitor,Electronics,250
4,P005,Webcam,Electronics,60
5,P006,Headphones,Electronics,120
6,P007,USB Hub,Electronics,35
7,P016,Cable Set,Electronics,30
8,P017,Power Bank,Electronics,50
9,P018,Speaker,Electronics,90


...

,product_id,product_name,total_revenue,revenue_rank
0,P001,Laptop,100800,1
1,P010,Standing Desk,68800,2
2,P020,Tablet,54600,3
3,P004,Monitor,44250,4
4,P009,Chair,28080,5
5,P019,Microphone,18590,6
6,P006,Headphones,17400,7
7,P018,Speaker,11520,8
8,P005,Webcam,10260,9
9,P017,Power Bank,8050,10


,month,revenue,running_total
0,None,405150,405150


,category,product_name,total_quantity_sold
0,Accessories,Phone Stand,137
1,Electronics,Monitor,177
2,Office,Desk Lamp,178
3,Stationery,Notebook,158


,product_id,product_name,total_revenue
0,P001,Laptop,100800
1,P004,Monitor,44250
2,P009,Chair,28080
3,P010,Standing Desk,68800
4,P020,Tablet,54600


,region,revenue,revenue_percentage
0,East,121636,30.10
1,West,117271,29.02
2,North,83444,20.65
3,South,81814,20.24


Q1. For Q20, how do you get the grand total to divide by — a subquery, or a window function with no PARTITION?

Answer: We can use a subquery or a window function without `PARTITION BY` to calculate the grand total.

Q2. Which of these were easy because you already did them in Pandas yesterday?

Answer: SELECT, WHERE, GROUP BY, JOIN, and aggregation were easy because I had already practiced them in Pandas.

Q3. Which query was the hardest to get right, and why?

Answer: JOIN was the hardest because matching the correct tables and columns requires care.

## Part B — the same analysis in Pandas AND SQL (the deliverable)

## TASK 02 Side-by-side: Pandas vs SQL

In [4]:

import pandas as pd
import sqlite3

# 1. Load your CSV files
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# 2. Convert dates correctly (DD-MM-YYYY)
sales["date"] = pd.to_datetime(
    sales["date"], format="%d-%m-%Y"
)

# 3. Solve using Pandas
full = sales.merge(
    customers, on="customer_id", how="left"
)

full = full.merge(
    products, on="product_id", how="left"
)

# Calculate revenue
full["revenue"] = full["quantity"] * full["unit_price"]

# Extract year and month
full["month"] = full["date"].dt.strftime("%Y-%m")

# Total revenue per region per month
pandas_result = (
    full.groupby(["region", "month"], as_index=False)
    ["revenue"]
    .sum()
    .rename(columns={"revenue": "total_revenue"})
    .sort_values(["region", "month"])
    .reset_index(drop=True)
)

print("PANDAS RESULT:")
print(pandas_result.to_string(index=False))


# 4. Create an SQLite database
conn = sqlite3.connect(":memory:")

# Convert dates to SQLite-friendly YYYY-MM-DD format
sales_sql = sales.copy()
sales_sql["date"] = sales_sql["date"].dt.strftime("%Y-%m-%d")

# Load the three tables into SQLite
sales_sql.to_sql("sales", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")


# 5. Solve the same problem using SQL
query = """
SELECT
    c.region,
    strftime('%Y-%m', s.date) AS month,
    SUM(s.quantity * p.unit_price) AS total_revenue
FROM sales AS s
JOIN customers AS c
    ON s.customer_id = c.customer_id
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY
    c.region,
    strftime('%Y-%m', s.date)
ORDER BY
    c.region,
    month;
"""

sql_result = pd.read_sql_query(query, conn)

print("\nSQL RESULT:")
print(sql_result.to_string(index=False))


# 6. Compare Pandas and SQL results
pd.testing.assert_frame_equal(
    pandas_result,
    sql_result,
    check_dtype=False,
    check_exact=False,
    rtol=1e-9,
    atol=1e-9
)

print("\nSUCCESS: Pandas and SQL results match!")

# 7. Close the database connection
conn.close()


PANDAS RESULT:
region   month  total_revenue
  East 2024-01          11506
  East 2024-02          11110
  East 2024-03          13911
  East 2024-04          10275
  East 2024-05           5976
  East 2024-06           6378
  East 2024-07           9568
  East 2024-08          12035
  East 2024-09          12640
  East 2024-10           8635
  East 2024-11           5029
  East 2024-12          14573
 North 2024-01           2975
 North 2024-02          12119
 North 2024-03           7269
 North 2024-04           8534
 North 2024-05          11601
 North 2024-06           6915
 North 2024-07           5573
 North 2024-08           3604
 North 2024-09           4852
 North 2024-10           6562
 North 2024-11           5831
 North 2024-12           7609
 South 2024-01           6114
 South 2024-02          12525
 South 2024-03          10415
 South 2024-04           3930
 South 2024-05           6320
 South 2024-06           3685
 South 2024-07           5078
 South 2024-08           

Q1. For this particular question, was Pandas or SQL more natural? Why?

Answer: Pandas was more natural because `merge()` and `groupby()` make data analysis simple and readable.

Q2. If the data were 500 million rows in a database, which would you use, and why?

Answer: I would use SQL because databases can process large datasets efficiently without loading all the data into Python memory.

Q3. Are there questions that are much easier in one than the other?

Answer: Yes. SQL is easier for database queries and joins, while Pandas is easier for data cleaning, visualization, and complex data analysis in Python.

## Part C — get comfortable with window functions

## TASK 03Window functions practice

In [5]:

import pandas as pd
import sqlite3

# 1. Load your CSV files
sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

# 2. Convert dates from DD-MM-YYYY to datetime
sales["date"] = pd.to_datetime(
    sales["date"], format="%d-%m-%Y"
)

# 3. Create SQLite database
conn = sqlite3.connect(":memory:")

# Convert dates to SQLite-compatible format
sales_sql = sales.copy()
sales_sql["date"] = sales_sql["date"].dt.strftime("%Y-%m-%d")

# Load CSV data into SQL tables
sales_sql.to_sql("sales", conn, index=False, if_exists="replace")
customers.to_sql("customers", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")


# TASK 1: Running total of revenue, ordered by date
query1 = """
SELECT
    s.order_id,
    s.date,
    SUM(s.quantity * p.unit_price) AS revenue,
    SUM(SUM(s.quantity * p.unit_price)) OVER (
        ORDER BY s.date, s.order_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total
FROM sales AS s
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY s.order_id, s.date
ORDER BY s.date, s.order_id;
"""

print("TASK 1: RUNNING TOTAL")
print(pd.read_sql_query(query1, conn).to_string(index=False))


# TASK 2: Top 10 customers by total spending
query2 = """
SELECT
    c.customer_id,
    c.customer_name,
    SUM(s.quantity * p.unit_price) AS total_spending
FROM sales AS s
JOIN customers AS c
    ON s.customer_id = c.customer_id
JOIN products AS p
    ON s.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_spending DESC
LIMIT 10;
"""

print("\nTASK 2: TOP 10 CUSTOMERS")
print(pd.read_sql_query(query2, conn).to_string(index=False))


# TASK 3: Rank customers by spending within each region
query3 = """
WITH customer_spending AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.region,
        SUM(s.quantity * p.unit_price) AS total_spending
    FROM sales AS s
    JOIN customers AS c
        ON s.customer_id = c.customer_id
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY c.customer_id, c.customer_name, c.region
)
SELECT
    customer_id,
    customer_name,
    region,
    total_spending,
    RANK() OVER (
        PARTITION BY region
        ORDER BY total_spending DESC
    ) AS regional_rank
FROM customer_spending
ORDER BY region, regional_rank;
"""

print("\nTASK 3: CUSTOMER RANK WITHIN EACH REGION")
print(pd.read_sql_query(query3, conn).to_string(index=False))


# TASK 4: Monthly revenue and previous month's revenue
query4 = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY strftime('%Y-%m', s.date)
)
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS previous_month_revenue
FROM monthly_revenue
ORDER BY month;
"""

print("\nTASK 4: MONTHLY REVENUE AND PREVIOUS MONTH")
print(pd.read_sql_query(query4, conn).to_string(index=False))


# TASK 5: Month-over-month revenue growth percentage
query5 = """
WITH monthly_revenue AS (
    SELECT
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales AS s
    JOIN products AS p
        ON s.product_id = p.product_id
    GROUP BY strftime('%Y-%m', s.date)
),
monthly_comparison AS (
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (ORDER BY month) AS previous_month_revenue
    FROM monthly_revenue
)
SELECT
    month,
    revenue,
    previous_month_revenue,
    ROUND(
        100.0 * (revenue - previous_month_revenue)
        / NULLIF(previous_month_revenue, 0),
        2
    ) AS growth_percentage
FROM monthly_comparison
ORDER BY month;
"""

print("\nTASK 5: MONTH-OVER-MONTH GROWTH")
print(pd.read_sql_query(query5, conn).to_string(index=False))


# Close database connection
conn.close()


TASK 1: RUNNING TOTAL
order_id       date  revenue  running_total
  O00281 2024-01-02     1250           1250
  O00673 2024-01-02      180           1430
  O00746 2024-01-02     1200           2630
  O00801 2024-01-02       24           2654
  O00004 2024-01-03      135           2789
  O00365 2024-01-03      800           3589
  O00475 2024-01-03      150           3739
  O00607 2024-01-03       18           3757
  O00745 2024-01-03     2000           5757
  O00203 2024-01-04       60           5817
  O00970 2024-01-04       25           5842
  O00690 2024-01-05      100           5942
  O00701 2024-01-05      150           6092
  O00310 2024-01-06     2400           8492
  O00739 2024-01-06      300           8792
  O00789 2024-01-06       45           8837
  O00938 2024-01-06     1000           9837
  O00315 2024-01-07      120           9957
  O00378 2024-01-07      750          10707
  O00133 2024-01-08      440          11147
  O00200 2024-01-08      450          11597
  O00440 2

Q1. What's the difference between RANK, DENSE_RANK, and ROW_NUMBER? When does it matter?

Answer:

- `RANK()` skips ranks after ties (1, 1, 3).
- `DENSE_RANK()` does not skip ranks (1, 1, 2).
- `ROW_NUMBER()` gives each row a unique number (1, 2, 3).

It matters when ranking customers or products with equal scores.

Q2. How does LAG let you compare a row to the one before it? What about LEAD?

Answer: `LAG()` retrieves a value from the previous row, while `LEAD()` retrieves a value from the next row. They help compare monthly revenue or track changes over time.

Q3. Why are window functions often cleaner than doing the same thing in Pandas?

Answer: Window functions perform ranking, running totals, and previous-row comparisons directly in SQL without requiring multiple intermediate DataFrames or merges.